In [2]:
pip install numpy

Note: you may need to restart the kernel to use updated packages.


In [50]:
import numpy as np

In [51]:
class Layer:
    def forward(self, input_value):
        raise NotImplementedError
        
    def backward(self, output_grad, alpha):
        raise NotImplementedError

class Loss:
    def calculate(self, y_true, y_pred):
        raise NotImplementedError
        
    def backward(self, y_true, y_pred):
        raise NotImplementedError

In [52]:
 
class Tanh(Layer):
    def __init__(self):
        self.input_value=None
        self.output_value=None
    
    def forward(self, input_value):
        self.input_value=input_value
        self.output_value=np.tanh(self.input_value)
        return self.output_value
    
    def backward(self, output_grad, alpha):
        activation_grad=1-np.power(self.output_value,2)
        return activation_grad * output_grad
    
class Sigmoid(Layer):
    def __init__(self):
        self.input_value=None
        self.output_value=None
    
    def forward(self, input_value):
        self.input_value=input_value
        self.output_value=1/(1+np.exp(-self.input_value))
        return self.output_value
    
    def backward(self, output_grad, alpha):
        activation_grad=self.output_value*(1-self.output_value)
        return activation_grad * output_grad


In [53]:
class MeanSquaredError(Loss):
    def calculate(self, y_true, y_pred):
        return np.mean(np.power(y_true - y_pred, 2))

    def backward(self, y_true, y_pred):
        return 2 * (y_pred - y_true) / y_true.size


class BinaryCrossEntropy(Loss):
    def calculate(self, y_true, y_pred):
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        
        term_0 = y_true * np.log(y_pred)
        term_1 = (1 - y_true) * np.log(1 - y_pred)
        return -np.mean(term_0 + term_1)

    def backward(self, y_true, y_pred):
        y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
        return (-(y_true / y_pred) + (1 - y_true) / (1 - y_pred)) / y_true.size

In [54]:
class Dense(Layer):
    def __init__(self, input_size, output_size):
        self.weights = np.random.randn(input_size, output_size) * np.sqrt(2.0 / input_size)
        self.biases = np.zeros((1, output_size))
        
        self.input_value = None

    def forward(self, input_value):
        self.input_value = input_value
        # Z = X * W + b
        return np.dot(self.input_value, self.weights) + self.biases

    def backward(self, output_grad, alpha ):
    
        weights_grad = np.dot(self.input_value.T, output_grad)
        biases_grad = np.sum(output_grad, axis=0, keepdims=True)
        
        #dX = dZ * W.T
        input_grad = np.dot(output_grad, self.weights.T)
        
        self.weights -= alpha * weights_grad
        self.biases -= alpha * biases_grad
        
        return input_grad

In [55]:
class NeuralNetwork:
    def __init__(self, loss_function: Loss):
        self.layers = []
        self.loss = loss_function

    def add(self, layer: Layer):
        self.layers.append(layer)

    def predict(self, input_value):
        output = input_value
        for layer in self.layers:
            output = layer.forward(output)
        return output

    def train(self, X_train, Y_train ,epochs, alpha):
        samples = len(X_train)
        
        for epoch in range(epochs):
            error = 0
            
            for x, y in zip(X_train, Y_train):
                x = x.reshape(1, -1)
                y = y.reshape(1, -1)
                
                output = self.predict(x)
                
                error += self.loss.calculate(y_true=y, y_pred=output)
                
                gradient = self.loss.backward(y_true=y, y_pred=output)
                for layer in reversed(self.layers):
                    gradient = layer.backward(gradient, alpha)
            
            error /= samples
            
            if (epoch + 1) % 100 == 0:
                print(f"epoch {epoch + 1}/{epochs}  error: {error:.4f}")

In [56]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
Y = np.array([[0], [1], [1], [0]])

model = NeuralNetwork(loss_function=MeanSquaredError())

model.add(Dense(input_size=2, output_size=3))
model.add(Tanh())                 
model.add(Dense(input_size=3, output_size=1))
model.add(Sigmoid())               

print("Training")
model.train(X, Y, epochs=1000, alpha=0.1)

print("\nTesting Predictions:")
for x, y in zip(X, Y):
    prediction = model.predict(x.reshape(1, -1))
    print(f"input: {x}  output: {y[0]}  pred: {prediction[0][0]:.4f}")


Training
epoch 100/1000  error: 0.2512
epoch 200/1000  error: 0.2470
epoch 300/1000  error: 0.2414
epoch 400/1000  error: 0.2087
epoch 500/1000  error: 0.1060
epoch 600/1000  error: 0.0282
epoch 700/1000  error: 0.0138
epoch 800/1000  error: 0.0088
epoch 900/1000  error: 0.0063
epoch 1000/1000  error: 0.0049

Testing Predictions:
input: [0 0]  output: 0  pred: 0.0613
input: [0 1]  output: 1  pred: 0.9150
input: [1 0]  output: 1  pred: 0.9258
input: [1 1]  output: 0  pred: 0.0555


In [57]:
def preprocess_mnist(X, Y):
    X = X.astype("float32") / 255.0
    
    X = X.reshape(X.shape[0], 784)
    
    num_classes = 10
    Y_one_hot = np.eye(num_classes)[Y]
    
    return X, Y_one_hot

In [58]:
from keras.datasets import mnist 

(x_train, y_train), (x_test, y_test) = mnist.load_data()

X_train, Y_train = preprocess_mnist(x_train, y_train)
X_test, Y_test = preprocess_mnist(x_test, y_test)

model = NeuralNetwork(loss_function=BinaryCrossEntropy())

model.add(Dense(784, 128))
model.add(Tanh()) 
model.add(Dense(128, 64))
model.add(Tanh()) 
model.add(Dense(64, 10))
model.add(Sigmoid()) 

print("Training")
model.train(X_train[:5000], Y_train[:5000], epochs=10, alpha=0.01)

correct = 0
for x, y in zip(X_test[:1000], y_test[:1000]):
    prediction = model.predict(x.reshape(1, -1))
    if np.argmax(prediction) == y:
        correct += 1

print(f"Test Accuracy: {correct / 1000 * 100:.2f}%")

Training
Test Accuracy: 90.00%
